# Step 4: Clean and Structure Performance Metrics

Transforms raw 2024 race results into an analysis-ready dataset:
1. Standardize column names and types
2. Clean driver and team names
3. Add sponsor identification
4. Calculate performance metrics
5. Validate the final dataset
6. Save cleaned data

## Step 1: Load and Standardize Column Names

In [12]:
import pandas as pd
import numpy as np
import re
import os

# Load raw data
df = pd.read_csv('../data/raw/race_results_2024_kaggle.csv')
print(f"Loaded {len(df)} rows")
print(f"Original columns: {df.columns.tolist()}")

Loaded 1398 rows
Original columns: ['year', 'race_num', 'track', 'track_type', 'fin', 'start', 'car_num', 'driver', 'manu', 'team_name', 'laps', 'laps_led', 'status', 'points', 'stage_1', 'stage_2', 'stage_3_or_duel', 'stage_points']


In [13]:
# Create a column mapping for consistent naming
# Adapted to actual column names in the Kaggle dataset
column_mapping = {
    'fin': 'Finish_Position',
    'Fin': 'Finish_Position',
    'Pos': 'Finish_Position',
    'Position': 'Finish_Position',
    'start': 'Start_Position',
    'St': 'Start_Position',
    'Start': 'Start_Position',
    'driver': 'Driver',
    'Driver': 'Driver',
    'team_name': 'Team',
    'Team': 'Team',
    'Owner': 'Team',
    'laps': 'Laps_Completed',
    'Laps': 'Laps_Completed',
    'laps_led': 'Laps_Led',
    'Led': 'Laps_Led',
    'Laps Led': 'Laps_Led',
    'track': 'Race_Name',
    'Race': 'Race_Name',
    'Track': 'Race_Name',
    'track_type': 'Track_Type',
    'manu': 'Manufacturer',
    'car_num': 'Car_Number',
    'race_num': 'Race_Number_Raw',
    'status': 'Status',
    'points': 'Points',
    'stage_1': 'Stage_1_Pos',
    'stage_2': 'Stage_2_Pos',
    'stage_points': 'Stage_Points',
}

df = df.rename(columns={col: column_mapping[col]
                        for col in df.columns
                        if col in column_mapping})

print(f"Standardized columns: {df.columns.tolist()}")

Standardized columns: ['year', 'Race_Number_Raw', 'Race_Name', 'Track_Type', 'Finish_Position', 'Start_Position', 'Car_Number', 'Driver', 'Manufacturer', 'Team', 'Laps_Completed', 'Laps_Led', 'Status', 'Points', 'Stage_1_Pos', 'Stage_2_Pos', 'stage_3_or_duel', 'Stage_Points']


In [14]:
def clean_position(pos):
    """Convert position to integer, handling special cases."""
    if pd.isna(pos):
        return np.nan
    pos_str = str(pos).strip()
    match = re.search(r'\d+', pos_str)
    if match:
        return int(match.group())
    return np.nan

df['Finish_Position'] = df['Finish_Position'].apply(clean_position)

if 'Start_Position' in df.columns:
    df['Start_Position'] = df['Start_Position'].apply(clean_position)

if 'Laps_Led' in df.columns:
    df['Laps_Led'] = pd.to_numeric(df['Laps_Led'], errors='coerce').fillna(0).astype(int)

if 'Stage_Points' in df.columns:
    df['Stage_Points'] = pd.to_numeric(df['Stage_Points'], errors='coerce').fillna(0)

print("Data types after conversion:")
print(df.dtypes)

Data types after conversion:
year                 int64
Race_Number_Raw      int64
Race_Name              str
Track_Type             str
Finish_Position      int64
Start_Position       int64
Car_Number           int64
Driver                 str
Manufacturer           str
Team                   str
Laps_Completed       int64
Laps_Led             int64
Status                 str
Points             float64
Stage_1_Pos        float64
Stage_2_Pos        float64
stage_3_or_duel    float64
Stage_Points         int64
dtype: object


## Step 2: Clean Driver and Team Names

In [15]:
print("Unique drivers:", df['Driver'].nunique())
print("\nSample driver names:")
for name in sorted(df['Driver'].unique())[:20]:
    print(f"  '{name}'")

Unique drivers: 62

Sample driver names:
  'A.J. Allmendinger'
  'Alex Bowman'
  'Anthony Alfredo'
  'Austin Cindric'
  'Austin Dillon'
  'Austin Hill'
  'B.J. McLeod'
  'Brad Keselowski'
  'Bubba Wallace'
  'Cameron Waters'
  'Carson Hocevar'
  'Chad Finchum'
  'Chase Briscoe'
  'Chase Elliott'
  'Chris Buescher'
  'Christopher Bell'
  'Cody Ware'
  'Corey Heim'
  'Corey LaJoie'
  'Daniel Hemric'


In [16]:
def clean_driver_name(name):
    """Standardize driver name format."""
    if pd.isna(name):
        return name
    name = ' '.join(str(name).split())
    name = re.sub(r'\s*\(i\)|\s*\*|\s*#\d+', '', name)
    name = name.title()
    return name.strip()

df['Driver'] = df['Driver'].apply(clean_driver_name)

# Correct known name variations
driver_corrections = {
    # Add corrections as you find them, e.g.:
    # 'Denny M Hamlin': 'Denny Hamlin',
}
df['Driver'] = df['Driver'].replace(driver_corrections)

print(f"Unique drivers after cleaning: {df['Driver'].nunique()}")

Unique drivers after cleaning: 62


In [17]:
def clean_team_name(name):
    """Standardize team name format."""
    if pd.isna(name):
        return name
    name = ' '.join(str(name).split())
    corrections = {
        'JGR': 'Joe Gibbs Racing',
        'HMS': 'Hendrick Motorsports',
        'Penske': 'Team Penske',
        'SHR': 'Stewart-Haas Racing',
        '23XI': '23XI Racing',
        'RCR': 'Richard Childress Racing',
    }
    for old, new in corrections.items():
        if name == old or name.lower() == old.lower():
            return new
    return name.strip()

df['Team'] = df['Team'].apply(clean_team_name)

print(f"Unique teams: {df['Team'].nunique()}")
print("\nTeam list:")
for team in sorted(df['Team'].unique()):
    print(f"  {team}")

Unique teams: 20

Team list:
  23XI Racing
  Beard Motorsports
  Front Row Motorsports
  Hendrick Motorsports
  JTG Daugherty Racing
  Joe Gibbs Racing
  Kaulig Racing
  Legacy Motor Club
  Live Fast Motorsports
  MBM Motorsports
  NY Racing Team
  RFK Racing
  Richard Childress Racing
  Rick Ware Racing
  Spire Motorsports
  Stewart-Haas Racing
  Team AmeriVet
  Team Penske
  Trackhouse Racing
  Wood Brothers Racing


## Step 3: Add Sponsor Identification

In [18]:
# Primary sponsor mapping: (Driver, Team) -> Sponsor
# Based on 2024 NASCAR Cup Series season sponsors
sponsor_mapping = {
    ('Denny Hamlin', 'Joe Gibbs Racing'): 'FedEx',
    ('Martin Truex Jr.', 'Joe Gibbs Racing'): 'Bass Pro Shops',
    ('Christopher Bell', 'Joe Gibbs Racing'): 'Rheem',
    ('Ty Gibbs', 'Joe Gibbs Racing'): 'Monster Energy',
    ('Kyle Larson', 'Hendrick Motorsports'): 'HendrickCars.com',
    ('Chase Elliott', 'Hendrick Motorsports'): 'NAPA Auto Parts',
    ('William Byron', 'Hendrick Motorsports'): 'Liberty University',
    ('Alex Bowman', 'Hendrick Motorsports'): 'Ally',
    ('Ryan Blaney', 'Team Penske'): 'Menards',
    ('Joey Logano', 'Team Penske'): 'Shell/Pennzoil',
    ('Austin Cindric', 'Team Penske'): 'Discount Tire',
    ('Bubba Wallace', '23XI Racing'): "McDonald's",
    ('Tyler Reddick', '23XI Racing'): 'Jordan Brand',
    ('Ross Chastain', 'Trackhouse Racing'): 'Busch Light',
    ('Daniel Suarez', 'Trackhouse Racing'): 'Freeway Insurance',
    ('Kyle Busch', 'Richard Childress Racing'): 'Cheddar\'s Scratch Kitchen',
    ('Austin Dillon', 'Richard Childress Racing'): 'Dow',
    ('Brad Keselowski', 'RFK Racing'): 'Dewalt',
    ('Chris Buescher', 'RFK Racing'): 'Fastenal',
    ('Michael Mcdowell', 'Front Row Motorsports'): 'Love\'s Travel Stops',
    ('Todd Gilliland', 'Front Row Motorsports'): 'Craftsman',
    ('Ricky Stenhouse Jr.', 'JTG Daugherty Racing'): 'Kroger',
    ('Noah Gragson', 'Stewart-Haas Racing'): 'Bass Pro Shops',
    ('Chase Briscoe', 'Stewart-Haas Racing'): 'Magical Vacation Planner',
    ('Kevin Harvick', 'Stewart-Haas Racing'): 'Hunt Brothers Pizza',
    ('Ryan Preece', 'Stewart-Haas Racing'): 'Sunseeker Resorts',
}

def get_sponsor(row):
    """Look up primary sponsor for driver/team combination."""
    key = (row['Driver'], row['Team'])
    return sponsor_mapping.get(key, 'Other/Unknown')

df['Primary_Sponsor'] = df.apply(get_sponsor, axis=1)

print("Sponsor distribution:")
print(df['Primary_Sponsor'].value_counts())

Sponsor distribution:
Primary_Sponsor
Other/Unknown                548
Jordan Brand                  37
NAPA Auto Parts               37
Ally                          37
Freeway Insurance             37
Shell/Pennzoil                37
Monster Energy                37
Fastenal                      37
Busch Light                   37
Sunseeker Resorts             37
Craftsman                     37
Dow                           37
Rheem                         37
Discount Tire                 37
FedEx                         37
Dewalt                        37
Liberty University            37
Magical Vacation Planner      37
McDonald's                    37
Love's Travel Stops           37
Menards                       37
Cheddar's Scratch Kitchen     37
Bass Pro Shops                37
HendrickCars.com              36
Name: count, dtype: int64


In [ ]:

# Define your 4 target sponsors from the project brief
target_sponsors = [
    'FedEx',           # Denny Hamlin / Joe Gibbs Racing  — Top tier
    'NAPA Auto Parts', # Chase Elliott / Hendrick Motorsports — Top tier
    "McDonald's",      # Bubba Wallace / 23XI Racing — Mid tier
    "Love's Travel Stops",  # Michael McDowell / Front Row Motorsports — Lower tier
]

df_targets = df[df['Primary_Sponsor'].isin(target_sponsors)].copy()
print(f"Filtered to target sponsors: {len(df_targets)} rows")
print(df_targets['Primary_Sponsor'].value_counts())


## Step 4: Calculate Performance Metrics

In [20]:
# Build race order from Race_Number_Raw (sequential race number)
# Race_Number_Raw of -1 indicates Duel races (qualifying); treat them as race 0
df['Race_Number'] = df['Race_Number_Raw'].apply(lambda x: 0 if x == -1 else x)

# Calculate positions gained/lost
if 'Start_Position' in df.columns:
    df['Positions_Gained'] = df['Start_Position'] - df['Finish_Position']

print("Added derived columns:")
cols = ['Race_Name', 'Race_Number', 'Driver', 'Start_Position', 'Finish_Position', 'Positions_Gained']
print(df[cols].head(10))

Added derived columns:
  Race_Name  Race_Number          Driver  Start_Position  Finish_Position  \
0   Daytona            0   Tyler Reddick              19                1   
1   Daytona            0   Chase Elliott               3                2   
2   Daytona            0     Alex Bowman               9                3   
3   Daytona            0  Carson Hocevar              20                4   
4   Daytona            0      Erik Jones              12                5   
5   Daytona            0   Daniel Suarez              10                6   
6   Daytona            0     Joey Logano               1                7   
7   Daytona            0        Ty Gibbs              15                8   
8   Daytona            0     Kyle Larson               2                9   
9   Daytona            0  Chris Buescher               8               10   

   Positions_Gained  
0                18  
1                 1  
2                 6  
3                16  
4                 7

In [21]:
# Season-level performance metrics per driver
driver_stats = df.groupby(['Driver', 'Team', 'Primary_Sponsor']).agg(
    Avg_Finish=('Finish_Position', 'mean'),
    Best_Finish=('Finish_Position', 'min'),
    Finish_StdDev=('Finish_Position', 'std'),
    Races_Started=('Finish_Position', 'count'),
    Total_Laps_Led=('Laps_Led', 'sum'),
    Avg_Laps_Led_Per_Race=('Laps_Led', 'mean'),
).round(2).reset_index()

# Top 5, Top 10, and win counts
top5 = df[df['Finish_Position'] <= 5].groupby('Driver').size()
top10 = df[df['Finish_Position'] <= 10].groupby('Driver').size()
wins = df[df['Finish_Position'] == 1].groupby('Driver').size()

driver_stats['Top_5_Finishes'] = driver_stats['Driver'].map(top5).fillna(0).astype(int)
driver_stats['Top_10_Finishes'] = driver_stats['Driver'].map(top10).fillna(0).astype(int)
driver_stats['Wins'] = driver_stats['Driver'].map(wins).fillna(0).astype(int)

driver_stats['Top_5_Rate'] = (driver_stats['Top_5_Finishes'] / driver_stats['Races_Started'] * 100).round(1)
driver_stats['Top_10_Rate'] = (driver_stats['Top_10_Finishes'] / driver_stats['Races_Started'] * 100).round(1)

print("=== Driver Season Statistics ===")
print(driver_stats.sort_values('Avg_Finish').to_string())

=== Driver Season Statistics ===
                  Driver                      Team            Primary_Sponsor  Avg_Finish  Best_Finish  Finish_StdDev  Races_Started  Total_Laps_Led  Avg_Laps_Led_Per_Race  Top_5_Finishes  Top_10_Finishes  Wins  Top_5_Rate  Top_10_Rate
34             Joey Hand                RFK Racing              Other/Unknown        4.00            4            NaN              1               7                   7.00               1                1     0       100.0        100.0
52       Parker Retzlaff         Beard Motorsports              Other/Unknown        7.00            7            NaN              1               0                   0.00               0                1     0         0.0        100.0
14         Chase Elliott      Hendrick Motorsports            NAPA Auto Parts       11.46            1           8.54             37             431                  11.65              12               20     1        32.4         54.1
16      Christopher Bel

In [22]:

# Aggregate to sponsor level
sponsor_stats = df.groupby('Primary_Sponsor').agg(
    Avg_Finish=('Finish_Position', 'mean'),
    Best_Finish=('Finish_Position', 'min'),
    Finish_StdDev=('Finish_Position', 'std'),
    Total_Laps_Led=('Laps_Led', 'sum'),
    Avg_Laps_Led=('Laps_Led', 'mean'),
    Num_Drivers=('Driver', 'nunique'),
    Total_Race_Entries=('Race_Number', 'count'),
).round(2).reset_index()

for sponsor in sponsor_stats['Primary_Sponsor']:
    sponsor_df = df[df['Primary_Sponsor'] == sponsor]
    total = len(sponsor_df)
    idx = sponsor_stats['Primary_Sponsor'] == sponsor
    sponsor_stats.loc[idx, 'Top_5_Rate'] = round(
        len(sponsor_df[sponsor_df['Finish_Position'] <= 5]) / total * 100, 1)
    sponsor_stats.loc[idx, 'Top_10_Rate'] = round(
        len(sponsor_df[sponsor_df['Finish_Position'] <= 10]) / total * 100, 1)
    sponsor_stats.loc[idx, 'Win_Rate'] = round(
        len(sponsor_df[sponsor_df['Finish_Position'] == 1]) / total * 100, 1)

print("=== Sponsor Performance Summary ===")
print(sponsor_stats.sort_values('Avg_Finish').to_string())


=== Sponsor Performance Summary ===
              Primary_Sponsor  Avg_Finish  Best_Finish  Finish_StdDev  Total_Laps_Led  Avg_Laps_Led  Num_Drivers  Total_Race_Entries  Top_5_Rate  Top_10_Rate  Win_Rate
19            NAPA Auto Parts       11.46            1           8.54             431         11.65            1                  37        32.4         54.1       2.7
21                      Rheem       12.49            1          12.97            1146         30.97            1                  37        43.2         64.9      10.8
11           HendrickCars.com       12.81            1          12.07            1720         47.78            1                  36        41.7         52.8      16.7
13         Liberty University       12.81            1          11.59             362          9.78            1                  37        35.1         59.5       8.1
12               Jordan Brand       12.86            1          11.37             598         16.16            1            

## Step 5: Validate the Final Dataset

In [ ]:

print("=== Final Validation ===\n")

# 1. Race completeness (2024 Cup Series has 36 points races)
expected_races = 36
points_races = df[df['Race_Number'] > 0]['Race_Name'].nunique()
print(f"Race completeness: {points_races}/{expected_races} ({points_races/expected_races*100:.0f}%)")

# 2. Target sponsor coverage
print(f"\nTarget sponsors found: {df[df['Primary_Sponsor'].isin(target_sponsors)]['Primary_Sponsor'].nunique()} of {len(target_sponsors)}")
for sponsor in target_sponsors:
    count = len(df[df['Primary_Sponsor'] == sponsor])
    print(f"  {sponsor}: {count} race entries")

# 3. Data quality checks
issues = []
bad_positions = df[df['Finish_Position'] > 50]
if len(bad_positions) > 0:
    issues.append(f"Positions > 50: {len(bad_positions)}")

for col in ['Race_Name', 'Driver', 'Team', 'Finish_Position']:
    missing = df[col].isnull().sum()
    if missing > 0:
        issues.append(f"Missing {col}: {missing}")

duplicates = df.duplicated(subset=['Race_Name', 'Race_Number', 'Driver']).sum()
if duplicates > 0:
    issues.append(f"Duplicate entries: {duplicates}")

if issues:
    print("\nData quality issues found:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("\nNo critical data quality issues found!")

# 4. Summary statistics
print("\n=== Dataset Summary ===")
print(f"Total rows: {len(df)}")
print(f"Unique races: {df['Race_Name'].nunique()}")
print(f"Unique drivers: {df['Driver'].nunique()}")
print(f"Unique teams: {df['Team'].nunique()}")
print(f"Unique sponsors: {df['Primary_Sponsor'].nunique()}")


## Step 6: Save Cleaned Data

In [24]:
os.makedirs('../data/processed', exist_ok=True)

# Clean master race results
clean_output = '../data/processed/race_results_2024_clean.csv'
df.to_csv(clean_output, index=False)
print(f"Saved clean race results to: {clean_output}")

# Driver statistics
driver_stats_output = '../data/processed/driver_stats_2024.csv'
driver_stats.to_csv(driver_stats_output, index=False)
print(f"Saved driver statistics to: {driver_stats_output}")

# Sponsor summary
sponsor_stats_output = '../data/processed/sponsor_stats_2024.csv'
sponsor_stats.to_csv(sponsor_stats_output, index=False)
print(f"Saved sponsor summary to: {sponsor_stats_output}")

# Target sponsors dataset
target_output = '../data/processed/race_results_targets_2024.csv'
df_targets.to_csv(target_output, index=False)
print(f"Saved target sponsors data to: {target_output}")

Saved clean race results to: ../data/processed/race_results_2024_clean.csv
Saved driver statistics to: ../data/processed/driver_stats_2024.csv
Saved sponsor summary to: ../data/processed/sponsor_stats_2024.csv
Saved target sponsors data to: ../data/processed/race_results_targets_2024.csv
